# M2aSoiling — Analysis Run (SRR rdtools)

Prasyarat di Google Drive `Cek PV String/`:
- `baseline/` — CSV baseline `{YYYY-MM}/{YYYY-MM-DD}.csv`
- `raw data input/` — Daily Rainfall PLTS IKN 2025/2026.xlsx, Report & Schedule Cleaning PLTS IKN.xlsx, List of DC Cables 0411.xls, **POA PLTS IKN 2025/2026.xlsx**, **Surface Albedo Forecast TMY NSRDB PLTS IKN.xlsx**, **PV Module Temperature PLTS IKN.xlsx** (untuk koreksi suhu SRR)
- `outputs/` — `m2_findings_{YYYYMMDD}.xlsx/.jsonl` hasil daily notebook (dipakai **mask availability M2e**: inverter-day uptime < 95% tidak dihitung sebagai soiling)

Tanpa file POA/albedo, POA jatuh ke pvlib clearsky → PR harian berisik → rawan `NoValidIntervalError`.

Jalankan cell berurut. Tiap varian analisis (WB03-10 / WB01-02) dipanggil lewat helper `run_soiling()` — argumen dikirim sebagai list ke subprocess, jadi path berspasi & pengulangan argumen aman.

> Analisis blended satu-site sengaja **tidak** disertakan: menggabung dua zona dengan kadens cleaning berbeda membuat deret PR terpecah menjadi interval < min_interval_length, sehingga SRR menghasilkan `NoValidIntervalError`. Soiling dianalisis per zona cleaning.

**Output tiap file** kini memuat sheet `CleaningImpact`: per event cleaning -> `sr_before`, `sr_after`, `sr_gain_pp`, `energy_recovered_kwh_per_day`, `rupiah_per_day`, `likely_cause`.

Cell terakhir run: **per-WB tunggal (WB01..WB10)** sekaligus (baseline dimuat sekali; kapasitas & biaya per-WB dari tabel bawaan).

Sheet tambahan: `DirectCleaningImpactPerString` (pre/post PR per string, otomatis), `PerInverterSRR` (sr/p_loss per inverter; aktif via `per_inverter_srr=True`), `PRDaily` (deret PR harian mentah), `SoilingRatio` (profil SR p50±CI), `MonthlySoilingLoss` (breakdown loss bulanan), `CleaningRecommendation` (prioritas area cleaning: deficit string vs sibling + p_loss inverter + loss historis; string mati ditandai `status=DEAD_OR_OFFLINE` dan dikeluarkan dari ranking), `AvailabilityMask` (inverter-day yang di-mask).

> **Satuan kolom (jangan dicampur):** `soiling_loss_pct` di `DirectCleaningImpact`/`DirectCleaningImpactPerString` = `(pr_after - pr_before) / pr_after` (perubahan RELATIF, sama seperti `pv_pipeline.yf_ratio_report`). `sr_gain_pp` di `CleaningImpact` = `sr_after - sr_before` (selisih dua rasio dalam POIN PERSEN). Keduanya bukan besaran yang sama.

**Cell 7** memplot tren **sawtooth** (PR harian + fit interval SRR + band CI) langsung dari xlsx hasil run — bisa diulang kapan pun tanpa run ulang.

In [ ]:
# Cell 1 - Mount Drive + FRESH clone (cegah clone bersarang saat re-run)
from google.colab import drive
drive.mount("/content/drive")

%cd /content
!rm -rf PVStringHeatmapCheck
!git clone https://github.com/ompltsikn/PVStringHeatmapCheck.git
%cd /content/PVStringHeatmapCheck

In [ ]:
# Cell 2 - Salin POA + albedo dari Drive ke "raw data input/" repo.
# POAProvider membacanya relatif terhadap cwd; tanpa ini POA fallback
# ke pvlib clearsky (PR berisik).
BASE = "/content/drive/MyDrive/Cek PV String"   # sesuaikan dengan path Drive Anda

!mkdir -p "raw data input"
!cp "{BASE}/raw data input/POA"*.xlsx "raw data input/" || echo "WARNING: file POA tidak ada di Drive -> POA pakai clearsky"
!cp "{BASE}/raw data input/Surface Albedo"*.xlsx "raw data input/" || echo "WARNING: file albedo tidak ada di Drive -> albedo statis 0.2"
!cp "{BASE}/raw data input/PV Module Temperature"*.xlsx "raw data input/" || echo "WARNING: file Tcell tidak ada -> koreksi suhu OFF (CF=1)"
!cp "{BASE}/raw data input/Ambient Temperature"*.xlsx "raw data input/" || echo "WARNING: Ambient Temp tidak ada -> SAPM fallback Tcell OFF"
!cp "{BASE}/raw data input/Wind Speed"*.xlsx "raw data input/" || echo "WARNING: Wind Speed tidak ada -> SAPM fallback Tcell OFF"
!cp "{BASE}/raw data input/Wind Direction"*.xlsx "raw data input/" || true
!ls -la "raw data input/"

In [ ]:
# Cell 3 - Helper run_soiling(): argumen dikirim sebagai LIST ke subprocess.
# Tanpa shell -> tak ada masalah quoting path berspasi / line-continuation.
# availability_dir="auto" -> pakai {BASE}/outputs (m2_findings_*.xlsx hasil
# daily notebook M2e): inverter-day uptime < 95% di-mask dari deret PR
# supaya outage parsial tidak terbaca sebagai soiling. Set None untuk off.
import subprocess
import sys


def run_soiling(*, cleaning_cost_idr=None, wb=None, capacity_kwp=None,
                per_wb=False, per_inverter_srr=False, availability_dir="auto"):
    if availability_dir == "auto":
        availability_dir = f"{BASE}/outputs"
    cmd = [
        sys.executable, "-u", "run_soiling_analysis.py",
        "--baseline-dir", f"{BASE}/baseline",
        "--rainfall-xlsx",
        f"{BASE}/raw data input/Daily Rainfall PLTS IKN 2025.xlsx",
        f"{BASE}/raw data input/Daily Rainfall PLTS IKN 2026.xlsx",
        "--cleaning-report-xlsx",
        f"{BASE}/raw data input/Report & Schedule Cleaning PLTS IKN.xlsx",
        "--dc-cable-xls", f"{BASE}/raw data input/List of DC Cables 0411.xls",
        "--clean-criterion", "precip_and_shift",
        "--precip-threshold-mm", "1.0",
        "--min-interval-length", "5",
        "--output-dir", f"{BASE}/outputs",
    ]
    if availability_dir:
        cmd += ["--availability-dir", availability_dir]
    if per_inverter_srr:
        cmd += ["--per-inverter-srr"]
    if per_wb:
        cmd += ["--per-wb"]
    else:
        cmd += ["--cleaning-cost-idr", str(cleaning_cost_idr)]
        if wb:
            cmd += ["--wb", *wb]
        if capacity_kwp is not None:
            cmd += ["--capacity-kwp", str(capacity_kwp)]
    print(">>", " ".join(cmd))
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()
    print("[exit code]", rc)
    return rc

In [ ]:
# Cell 4 - Kelompok WB03-10 (water truck solar; kapasitas ~58.000 kWp)
run_soiling(
    wb=["WB03", "WB04", "WB05", "WB06", "WB07", "WB08", "WB09", "WB10"],
    capacity_kwp=58000,
    cleaning_cost_idr=55000000,
    per_inverter_srr=True,
)

In [ ]:
# Cell 5 - Kelompok WB01-02 (gravitasi + pompa; kapasitas ~13.500 kWp)
run_soiling(
    wb=["WB01", "WB02"],
    capacity_kwp=13500,
    cleaning_cost_idr=1165000,
    per_inverter_srr=True,
)

In [ ]:
# Cell 6 - Per-WB tunggal WB01..WB10 (baseline dimuat SEKALI).
# Kapasitas & cleaning cost per WB dari tabel bawaan run_soiling_analysis.py
# (PER_WB_CAPACITY_KWP / PER_WB_CLEANING_COST_IDR). Menghasilkan 10 file:
# soiling_srr_<periode>_WB01.xlsx .. _WB10.xlsx, masing-masing dengan sheet
# CleaningImpact (sr_gain_pp + kWh/rupiah dipulihkan per event cleaning).
run_soiling(per_wb=True)

In [ ]:
# Cell 7 - Plot tren SAWTOOTH dari xlsx hasil run (tanpa run ulang, tanpa
# rdtools): PR harian (PRDaily) + profil SR p50±CI (SoilingRatio) + interval
# SRR (CleaningEvents). Gigi turun = soiling menumpuk; lompatan = cleaning/
# hujan. Merah tebal = interval valid; abu tipis = tidak valid. PNG disimpan
# di folder outputs. File hasil run lama (tanpa sheet PRDaily) dilewati.
import glob
import os

import matplotlib.pyplot as plt
import pandas as pd

OUT_DIR = f"{BASE}/outputs"
plot_files = sorted(glob.glob(os.path.join(OUT_DIR, "soiling_srr_*.xlsx")))
print(f"[plot] {len(plot_files)} file soiling_srr ditemukan di {OUT_DIR}")

for path in plot_files:
    tag = os.path.splitext(os.path.basename(path))[0]
    xl = pd.ExcelFile(path)
    if "PRDaily" not in xl.sheet_names:
        print(f"[plot] SKIP {tag}: tidak ada sheet PRDaily "
              "(hasil run lama -> jalankan ulang Cell 4/5/6).")
        continue
    pr = pd.read_excel(path, sheet_name="PRDaily", parse_dates=["date"])

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.scatter(pr["date"], pr["pr"], s=9, color="0.65", zorder=2,
               label="PR harian (ternormalisasi POA)")

    if "SoilingRatio" in xl.sheet_names:
        srp = pd.read_excel(path, sheet_name="SoilingRatio",
                            parse_dates=["date"])
        ax.fill_between(srp["date"], srp["sr_ci_lower"], srp["sr_ci_upper"],
                        color="tab:blue", alpha=0.15, zorder=1,
                        label="SR CI (Monte Carlo)")
        ax.plot(srp["date"], srp["sr_p50"], color="tab:blue", lw=1.3,
                zorder=3, label="SR p50 (SRR)")

    if "CleaningEvents" in xl.sheet_names:
        ce = pd.read_excel(path, sheet_name="CleaningEvents",
                           parse_dates=["start", "end"])
        for _, r in ce.sort_values("start").iterrows():
            valid = bool(r.get("valid", False))
            ax.plot([r["start"], r["end"]],
                    [r["inferred_start_loss"], r["inferred_end_loss"]],
                    color="tab:red" if valid else "0.8",
                    lw=2.5 if valid else 1.0, zorder=4 if valid else 1)

    q = pr["pr"].dropna()
    if not q.empty:
        ax.set_ylim(max(0.5, q.quantile(0.02) - 0.06),
                    min(1.5, q.quantile(0.98) + 0.08))
    ax.set_ylabel("PR / soiling ratio")
    ax.set_title(f"Sawtooth soiling - {tag}")
    ax.legend(loc="lower left", ncol=3, fontsize=9)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    png = os.path.join(OUT_DIR, f"{tag}_sawtooth.png")
    fig.savefig(png, dpi=150)
    plt.show()
    print(f"[plot] PNG: {png}")

    if "MonthlySoilingLoss" in xl.sheet_names:
        ml = pd.read_excel(path, sheet_name="MonthlySoilingLoss")
        print(f"[plot] {tag} - soiling loss bulanan:")
        print(ml[["month", "n_days", "sr_p50", "p_loss_pct",
                  "energy_lost_kwh_est"]].round(4).to_string(index=False))